In [35]:
import pandas as pd
import numpy as np
import os

print("Libraries imported successfully")

Libraries imported successfully


## Configuration

In [36]:
# LNER station coordinates
LNER_STATIONS = {
    'Kings Cross London': (51.5308, -0.1238),
    'Newcastle Central': (54.9683, -1.6174),
    'Edinburgh Waverley': (55.9520, -3.1883),
    'York': (53.9579, -1.0934),
    'Leeds': (53.7955, -1.5481),
    'Doncaster': (53.5226, -1.1391),
    'Peterborough': (52.5746, -0.2503),
    'Durham': (54.7792, -1.5832),
}

# Stadium data (23 teams)
STADIUM_DATA = {
    'Arsenal': {'name': 'Emirates Stadium', 'city': 'London', 'lat': 51.5549, 'lon': -0.1084, 'capacity': 60704},
    'Aston Villa': {'name': 'Villa Park', 'city': 'Birmingham', 'lat': 52.5092, 'lon': -1.8851, 'capacity': 42640},
    'Bournemouth': {'name': 'Vitality Stadium', 'city': 'Bournemouth', 'lat': 50.7352, 'lon': -1.8382, 'capacity': 11364},
    'Brentford': {'name': 'Gtech Community Stadium', 'city': 'London', 'lat': 51.4908, 'lon': -0.2887, 'capacity': 17250},
    'Brighton': {'name': 'American Express Stadium', 'city': 'Brighton', 'lat': 50.8609, 'lon': -0.0831, 'capacity': 31800},
    'Chelsea': {'name': 'Stamford Bridge', 'city': 'London', 'lat': 51.4817, 'lon': -0.1910, 'capacity': 40341},
    'Crystal Palace': {'name': 'Selhurst Park', 'city': 'London', 'lat': 51.3983, 'lon': -0.0854, 'capacity': 25486},
    'Everton': {'name': 'Goodison Park', 'city': 'Liverpool', 'lat': 53.4387, 'lon': -2.9663, 'capacity': 39414},
    'Fulham': {'name': 'Craven Cottage', 'city': 'London', 'lat': 51.4749, 'lon': -0.2217, 'capacity': 29600},
    'Ipswich': {'name': 'Portman Road', 'city': 'Ipswich', 'lat': 52.0550, 'lon': 1.1447, 'capacity': 30311},
    'Leicester': {'name': 'King Power Stadium', 'city': 'Leicester', 'lat': 52.6204, 'lon': -1.1420, 'capacity': 32261},
    'Liverpool': {'name': 'Anfield', 'city': 'Liverpool', 'lat': 53.4308, 'lon': -2.9608, 'capacity': 61276},
    'Manchester City': {'name': 'Etihad Stadium', 'city': 'Manchester', 'lat': 53.4831, 'lon': -2.2004, 'capacity': 53400},
    'Manchester United': {'name': 'Old Trafford', 'city': 'Manchester', 'lat': 53.4631, 'lon': -2.2913, 'capacity': 74310},
    'Newcastle': {'name': "St James' Park", 'city': 'Newcastle', 'lat': 54.9756, 'lon': -1.6217, 'capacity': 52305},
    'Nottingham Forest': {'name': 'City Ground', 'city': 'Nottingham', 'lat': 52.9400, 'lon': -1.1327, 'capacity': 30445},
    'Southampton': {'name': "St Mary's Stadium", 'city': 'Southampton', 'lat': 50.9059, 'lon': -1.3909, 'capacity': 32384},
    'Tottenham': {'name': 'Tottenham Hotspur Stadium', 'city': 'London', 'lat': 51.6043, 'lon': -0.0664, 'capacity': 62850},
    'West Ham': {'name': 'London Stadium', 'city': 'London', 'lat': 51.5386, 'lon': -0.0163, 'capacity': 62500},
    'Wolverhampton Wanderers': {'name': 'Molineux Stadium', 'city': 'Wolverhampton', 'lat': 52.5902, 'lon': -2.1305, 'capacity': 32050},
    'Burnley': {'name': 'Turf Moor', 'city': 'Burnley', 'lat': 53.7889, 'lon': -2.2302, 'capacity': 21944},
    'Luton': {'name': 'Kenilworth Road', 'city': 'Luton', 'lat': 51.8844, 'lon': -0.4317, 'capacity': 10356},
    'Sheffield Utd': {'name': 'Bramall Lane', 'city': 'Sheffield', 'lat': 53.3704, 'lon': -1.4709, 'capacity': 32050},
}

DERBY_MATCHES = {
    'North London': [('Arsenal', 'Tottenham'), ('Tottenham', 'Arsenal')],
    'Manchester': [('Manchester City', 'Manchester United'), ('Manchester United', 'Manchester City')],
    'Merseyside': [('Everton', 'Liverpool'), ('Liverpool', 'Everton')]
}

LONDON_CLUBS = ['Arsenal', 'Brentford', 'Chelsea', 'Crystal Palace', 'Fulham', 'Tottenham', 'West Ham']
TOP_6_CLUBS = ['Arsenal', 'Chelsea', 'Liverpool', 'Manchester City', 'Manchester United', 'Tottenham']

# DISTANCE THRESHOLD - Change this value to adjust
DISTANCE_THRESHOLD_KM = 50.0

print(f"Configuration complete - Using {DISTANCE_THRESHOLD_KM}km threshold")

Configuration complete - Using 50.0km threshold


## Helper Functions

In [37]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    dlat, dlon = lat2_rad - lat1_rad, lon2_rad - lon1_rad
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

def find_nearest_lner_station(stadium_lat, stadium_lon):
    min_distance, nearest_station = float('inf'), None
    for station_name, (station_lat, station_lon) in LNER_STATIONS.items():
        distance = haversine_distance(stadium_lat, stadium_lon, station_lat, station_lon)
        if distance < min_distance:
            min_distance, nearest_station = distance, station_name
    return nearest_station, min_distance

def standardize_team_name(name):
    mapping = {'Manchester Utd': 'Manchester United', 'Nottingham': 'Nottingham Forest',
               'Wolves': 'Wolverhampton Wanderers'}
    return mapping.get(name, name)

def is_derby_match(home_team, away_team):
    for derby_type, matchups in DERBY_MATCHES.items():
        if (home_team, away_team) in matchups:
            return True, derby_type
    return (True, 'London') if home_team in LONDON_CLUBS and away_team in LONDON_CLUBS else (False, None)

def is_top6_clash(home_team, away_team):
    return home_team in TOP_6_CLUBS and away_team in TOP_6_CLUBS

print("Helper functions defined")

Helper functions defined


## Load Data

In [38]:
input_file = '/content/Database - Overview - England Premier League - LS.csv'

df = pd.read_csv(input_file)
print(f"Loaded: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded: 760 rows
Columns: ['id', 'matchDate', 'Country', 'League', 'Season', 'homeTeam', 'awayTeam', 'referee', 'FTHG', 'FTAG', 'FTR']


,id,matchDate,Country,League,Season,homeTeam,awayTeam,referee,FTHG,FTAG,FTR
0,14540,25-05-25 17:00,England,Premier League,2024/2025,Bournemouth,Leicester,Lewis Smith,2,0,H
1,14541,25-05-25 17:00,England,Premier League,2024/2025,Fulham,Manchester City,Andy Madley,0,2,A
2,14542,25-05-25 17:00,England,Premier League,2024/2025,Ipswich,West Ham,Tim Robinson,1,3,A


## Data Processing

In [39]:
# Standardize team names
df['homeTeam'] = df['homeTeam'].apply(standardize_team_name)
df['awayTeam'] = df['awayTeam'].apply(standardize_team_name)

# Parse dates
df['matchDate'] = pd.to_datetime(df['matchDate'], format='%d-%m-%y %H:%M')

# Extract temporal features
df['hour'] = df['matchDate'].dt.hour
df['day_of_week'] = df['matchDate'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6])

print("Temporal features created")

Temporal features created


In [40]:
# Add stadium information
df['stadium_name'] = df['homeTeam'].map(lambda x: STADIUM_DATA[x]['name'])
df['stadium_city'] = df['homeTeam'].map(lambda x: STADIUM_DATA[x]['city'])
df['stadium_lat'] = df['homeTeam'].map(lambda x: STADIUM_DATA[x]['lat'])
df['stadium_lon'] = df['homeTeam'].map(lambda x: STADIUM_DATA[x]['lon'])
df['stadium_capacity'] = df['homeTeam'].map(lambda x: STADIUM_DATA[x]['capacity'])

print("Stadium information added")

Stadium information added


In [41]:
# Calculate nearest LNER station
nearest_stations = []
distances = []

for idx, row in df.iterrows():
    nearest, distance = find_nearest_lner_station(row['stadium_lat'], row['stadium_lon'])
    nearest_stations.append(nearest)
    distances.append(distance)

df['nearest_lner_station'] = nearest_stations
df['distance_to_lner_km'] = distances

# Apply threshold
df['is_lner_relevant'] = df['distance_to_lner_km'] <= DISTANCE_THRESHOLD_KM

print(f"LNER mapping complete with {DISTANCE_THRESHOLD_KM}km threshold")
print(f"LNER-relevant matches: {df['is_lner_relevant'].sum()} / {len(df)}")

LNER mapping complete with 50.0km threshold
LNER-relevant matches: 361 / 760


In [42]:
# Add match classifications
derby_info = df.apply(lambda row: is_derby_match(row['homeTeam'], row['awayTeam']), axis=1)
df['is_derby'] = derby_info.apply(lambda x: x[0])
df['derby_type'] = derby_info.apply(lambda x: x[1])
df['is_top6_clash'] = df.apply(lambda row: is_top6_clash(row['homeTeam'], row['awayTeam']), axis=1)

df['is_high_risk_match'] = (
    df['is_derby'] | df['is_top6_clash'] |
    (df['is_weekend'] & (df['stadium_capacity'] > 50000)) |
    ((df['hour'] >= 17) & (df['stadium_city'] == 'London'))
)

print("Match classifications added")

Match classifications added


In [43]:
# Estimate attendance
df['estimated_attendance_pct'] = 0.85
df.loc[df['is_derby'], 'estimated_attendance_pct'] += 0.10
df.loc[df['is_top6_clash'], 'estimated_attendance_pct'] += 0.05
df.loc[df['is_weekend'], 'estimated_attendance_pct'] += 0.05
df['estimated_attendance_pct'] = df['estimated_attendance_pct'].clip(upper=1.0)
df['estimated_attendance'] = (df['estimated_attendance_pct'] * df['stadium_capacity']).astype(int)

print("Attendance estimation complete")

Attendance estimation complete


## Create Minimal Dataset (7 Columns)

In [44]:
# Filter to LNER-relevant matches only
df_lner = df[df['is_lner_relevant']].copy()

# Select only 7 essential columns
columns_minimal = [
    'stadium_name',
    'stadium_city',
    'stadium_lat',
    'stadium_lon',
    'stadium_capacity',
    'nearest_lner_station',
    'estimated_attendance',
]

df_minimal = df_lner[columns_minimal].copy()

print(f"Minimal dataset created: {len(df_minimal)} rows × {len(df_minimal.columns)} columns")
df_minimal.head()

Minimal dataset created: 361 rows × 7 columns


,stadium_name,stadium_city,stadium_lat,stadium_lon,stadium_capacity,nearest_lner_station,estimated_attendance
1,Craven Cottage,London,51.4749,-0.2217,29600,Kings Cross London,26640
5,St James' Park,Newcastle,54.9756,-1.6217,52305,Newcastle Central,47074
8,Tottenham Hotspur Stadium,London,51.6043,-0.0664,62850,Kings Cross London,56565
10,Selhurst Park,London,51.3983,-0.0854,25486,Kings Cross London,21663
13,Emirates Stadium,London,51.5549,-0.1084,60704,Kings Cross London,54633


## Save to CSV

In [45]:
output_file = 'Clean dataset.csv'
df_minimal.to_csv(output_file, index=False)

print(f"\n{'='*80}")
print("SUCCESS: File created with 50km threshold")
print('='*80)
print(f"Filename: {output_file}")
print(f"Rows: {len(df_minimal):,}")
print(f"Columns: {len(df_minimal.columns)}")
print(f"Size: {os.path.getsize(output_file):,} bytes")


SUCCESS: File created with 50km threshold
Filename: Clean dataset.csv
Rows: 361
Columns: 7
Size: 25,398 bytes


## Summary Statistics

In [46]:
print(f"\n📍 LNER Stations in dataset:")
station_counts = df_lner['nearest_lner_station'].value_counts()
for station, count in station_counts.items():
    pct = count / len(df_lner) * 100
    print(f"   {station:<25} {count:>3} matches ({pct:>5.1f}%)")

print(f"\nTotal matches: {len(df)}")
print(f"LNER-relevant (50km): {len(df_lner)}")
print(f"Percentage: {len(df_lner)/len(df)*100:.1f}%")

print("\n🎯 NEW ADDITIONS (vs 20km threshold):")
print("   + Doncaster station (Sheffield United)")
print("   + Leeds station (Burnley)")


📍 LNER Stations in dataset:
   Kings Cross London        285 matches ( 78.9%)
   Newcastle Central          38 matches ( 10.5%)
   Leeds                      19 matches (  5.3%)
   Doncaster                  19 matches (  5.3%)

Total matches: 760
LNER-relevant (50km): 361
Percentage: 47.5%

🎯 NEW ADDITIONS (vs 20km threshold):
   + Doncaster station (Sheffield United)
   + Leeds station (Burnley)


## Download File (Google Colab Only)

In [47]:
from google.colab import files
files.download(output_file)
print(f"Downloaded: {output_file}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: Clean dataset.csv
